# Eksperimen & Benchmark Linear Regression from Scratch

Notebook ini mengimplementasikan dan menguji model `linear_regression` (from scratch) sesuai spesifikasi tugas:
1. **a. Gradient Descent (Batch & Mini-Batch) + Normal Equation** sebagai pembanding.
2. **b. Regularisasi L2 (Ridge) & L1 (Lasso)**.
3. **c. Standarisasi fitur numerik** sebelum pelatihan, dan **bias/intercept tidak ikut diregularisasi**.
4. **d. Perbandingan dengan `LinearRegression`, `Ridge`, dan `Lasso` dari Scikit-Learn** menggunakan metrik **MAE, MSE, RMSE, dan R²**, disertai visualisasi **Predicted vs Actual Plot** dan **Residual Plot**.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import class Linear Regression from scratch
from linear_regression import linear_regression

# Scikit-Learn imports untuk pembanding
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression as SklearnLinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
print("Semua modul berhasil di-import!")

## 1. Load Data

In [ ]:
df = pd.read_csv('train.csv')
print(f"Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

## 2. Preprocessing & Standarisasi Fitur Numerik (Ketentuan c)

Tahapan:
1. Drop kolom `id`
2. Imputasi missing values: median (numerik) dan modus (kategorikal)
3. One-Hot Encoding untuk fitur kategorikal
4. Train-Test Split (80% Train, 20% Test)
5. **Standarisasi khusus fitur numerik** (Z-Score Standardization: $\frac{X - \mu}{\sigma}$) menggunakan statistik dari data training agar tidak terjadi *data leakage*.

In [ ]:
# Pisahkan Target dan Fitur
X = df.drop(columns=['id', 'target']).copy()
y = df['target'].values

# Identifikasi kolom numerik dan kategorikal
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Imputasi missing values
for col in numeric_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].median())

for col in cat_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])

# One-Hot Encoding fitur kategorikal
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=float)

# Train - Test Split (sebelum standarisasi)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

X_train_df = X_train_df.copy()
X_test_df = X_test_df.copy()

# Standarisasi HANYA pada fitur numerik
mean_num = X_train_df[numeric_cols].mean(axis=0)
std_num = X_train_df[numeric_cols].std(axis=0)
std_num[std_num == 0] = 1.0

X_train_df[numeric_cols] = (X_train_df[numeric_cols] - mean_num) / std_num
X_test_df[numeric_cols] = (X_test_df[numeric_cols] - mean_num) / std_num

X_train_scaled = X_train_df.values
X_test_scaled = X_test_df.values

print(f"Jumlah fitur numerik yang distandarisasi: {len(numeric_cols)}")
print(f"Total fitur keseluruhan: {X_train_scaled.shape[1]}")
print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_test shape:  {X_test_scaled.shape}")

## 3. Pelatihan Model Scratch vs Scikit-Learn (Ketentuan a & b)

Kita melatih semua varian model **From Scratch** dan membandingkannya dengan **Scikit-Learn**:
1. **Scratch - Linear Regression (Batch GD, Tanpa Regularisasi)**
2. **Scratch - Linear Regression (Mini-Batch GD, Tanpa Regularisasi)**
3. **Scratch - Normal Equation (Solusi Analitik)**
4. **Scratch - L2 Regularization (Ridge)**
5. **Scratch - L1 Regularization (Lasso)**
6. **Scikit-Learn - LinearRegression**
7. **Scikit-Learn - Ridge (L2)**
8. **Scikit-Learn - Lasso (L1)**

In [ ]:
lmbda_val = 0.05

# --- MODEL FROM SCRATCH ---
# 1. Scratch OLS (Batch Gradient Descent)
scratch_ols = linear_regression(penalty=None, method='gradient_descent')
scratch_ols.fit(X_train_scaled, y_train, max_iter=10000, learning_rate=0.005)

# 2. Scratch OLS (Mini-Batch Gradient Descent)
scratch_minibatch = linear_regression(penalty=None, method='gradient_descent', batch_size=256)
scratch_minibatch.fit(X_train_scaled, y_train, max_iter=10000, learning_rate=0.005)

# 3. Scratch Normal Equation
scratch_normal = linear_regression(penalty=None, method='normal_equation')
scratch_normal.fit(X_train_scaled, y_train)

# 4. Scratch L2 Regularization (Ridge)
scratch_l2 = linear_regression(penalty='l2', lmbda=lmbda_val, method='gradient_descent')
scratch_l2.fit(X_train_scaled, y_train, max_iter=10000, learning_rate=0.005)

# 5. Scratch L1 Regularization (Lasso)
scratch_l1 = linear_regression(penalty='l1', lmbda=lmbda_val, method='gradient_descent')
scratch_l1.fit(X_train_scaled, y_train, max_iter=10000, learning_rate=0.005)

# --- MODEL SCIKIT-LEARN ---
# 6. Scikit-Learn LinearRegression
sklearn_lr = SklearnLinearRegression()
sklearn_lr.fit(X_train_scaled, y_train)

# 7. Scikit-Learn Ridge (L2)
# Catatan: Objektif Ridge di Sklearn meminimalkan ||y - Xw||^2_2 + alpha * ||w||^2_2
sklearn_ridge = Ridge(alpha=lmbda_val * len(X_train_scaled))
sklearn_ridge.fit(X_train_scaled, y_train)

# 8. Scikit-Learn Lasso (L1)
# Catatan: Objektif Lasso di Sklearn meminimalkan (1 / 2n) ||y - Xw||^2_2 + alpha * ||w||_1
sklearn_lasso = Lasso(alpha=lmbda_val / 2, max_iter=10000)
sklearn_lasso.fit(X_train_scaled, y_train)

print("Semua 8 model berhasil dilatih!")

### Kurva Konvergensi Loss (Scratch Models)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(scratch_ols.loss_history, label='Scratch OLS (Batch GD)', lw=2)
plt.plot(scratch_l2.loss_history, label='Scratch L2 (Ridge)', lw=2, linestyle='--')
plt.plot(scratch_l1.loss_history, label='Scratch L1 (Lasso)', lw=2, linestyle=':')
plt.plot(scratch_minibatch.loss_history, label='Scratch Mini-Batch GD', alpha=0.4)
plt.title('Penurunan MSE Loss selama Pelatihan Gradient Descent')
plt.xlabel('Iterasi')
plt.ylabel('MSE Loss')
plt.ylim(0, 50)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## 4. Perbandingan Metrik Regresi: MAE, MSE, RMSE, dan R² (Ketentuan d)

In [ ]:
predictions = {
    'Scratch OLS (Batch GD)': scratch_ols.predict(X_test_scaled),
    'Scratch OLS (Mini-Batch GD)': scratch_minibatch.predict(X_test_scaled),
    'Scratch OLS (Normal Eq)': scratch_normal.predict(X_test_scaled),
    'Scratch L2 (Ridge)': scratch_l2.predict(X_test_scaled),
    'Scratch L1 (Lasso)': scratch_l1.predict(X_test_scaled),
    'Sklearn LinearRegression': sklearn_lr.predict(X_test_scaled),
    'Sklearn Ridge (L2)': sklearn_ridge.predict(X_test_scaled),
    'Sklearn Lasso (L1)': sklearn_lasso.predict(X_test_scaled),
}

metrics_list = []
for name, y_pred in predictions.items():
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    metrics_list.append({
        'Model': name,
        'MAE': round(mae, 4),
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'R² Score': round(r2, 4)
    })

df_metrics = pd.DataFrame(metrics_list)
df_metrics

## 5. Visualisasi 1: Predicted vs Actual Plot (Scratch vs Scikit-Learn)

Membandingkan hasil prediksi masing-masing varian model Scratch (OLS, L2, L1) dengan pasangan Scikit-Learn-nya (`LinearRegression`, `Ridge`, `Lasso`).

In [ ]:
compare_keys = [
    'Scratch OLS (Batch GD)', 'Sklearn LinearRegression',
    'Scratch L2 (Ridge)', 'Sklearn Ridge (L2)',
    'Scratch L1 (Lasso)', 'Sklearn Lasso (L1)'
]

fig, axes = plt.subplots(3, 2, figsize=(14, 16))
axes = axes.flatten()

for i, name in enumerate(compare_keys):
    ax = axes[i]
    y_pred = predictions[name]
    r2_val = df_metrics.loc[df_metrics['Model'] == name, 'R² Score'].values[0]
    rmse_val = df_metrics.loc[df_metrics['Model'] == name, 'RMSE'].values[0]
    
    color = 'royalblue' if 'Scratch' in name else 'seagreen'
    ax.scatter(y_test, y_pred, alpha=0.3, color=color, edgecolors='none')
    
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal (y = ŷ)')
    ax.set_title(f'{name}\n(RMSE = {rmse_val}, R² = {r2_val})')
    ax.set_xlabel('Actual Target (y)')
    ax.set_ylabel('Predicted Target (ŷ)')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

plt.suptitle('Predicted vs Actual: Scratch vs Scikit-Learn', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 6. Visualisasi 2: Residual Plot (Scratch vs Scikit-Learn)

Residual ($y - \hat{y}$) diplot terhadap nilai prediksi ($\hat{y}$). Penyebaran acak di sekitar garis horizontal $0$ menunjukkan tidak adanya bias sistematis.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 16))
axes = axes.flatten()

for i, name in enumerate(compare_keys):
    ax = axes[i]
    y_pred = predictions[name]
    residuals = y_test - y_pred
    
    color = 'darkorange' if 'Scratch' in name else 'mediumpurple'
    ax.scatter(y_pred, residuals, alpha=0.3, color=color, edgecolors='none')
    ax.axhline(0, color='red', linestyle='--', lw=2, label='Zero Error Line')
    ax.set_title(f'Residual Plot: {name}')
    ax.set_xlabel('Predicted Target (ŷ)')
    ax.set_ylabel('Residuals (y - ŷ)')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

plt.suptitle('Residual Plots: Scratch vs Scikit-Learn', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 7. Simpan & Muat Model (Save & Load via Pickle)

In [ ]:
# Simpan model terbaik scratch ke file .pkl
scratch_l2.save('model_linear_regression_l2.pkl')

# Muat kembali model dan uji prediksi
loaded_model = linear_regression.load('model_linear_regression_l2.pkl')
sample_preds = loaded_model.predict(X_test_scaled[:5])
print("Prediksi 5 sampel pertama (Loaded Model):", np.round(sample_preds, 2))
print("Nilai aktual 5 sampel pertama          :", y_test[:5])